In [3]:
import pandas as pd
from transformers import AutoTokenizer, RobertaModel, AutoConfig
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score
import torch

import utils
import const
import models

# Read and Prepare Data

In [2]:
relations_df = utils.get_relations()
relations_df.head()

,docid,verb1,verb2,eiid1,eiid2,relation
0,WSJ_20130322_159,apologized,happened,E1,E5,VAGUE
1,WSJ_20130322_159,apologized,wrapped,E1,E6,BEFORE
2,WSJ_20130322_159,apologized,seemed,E1,E10,BEFORE
3,WSJ_20130322_159,apologized,yield,E1,E11,VAGUE
4,WSJ_20130322_159,happened,wrapped,E5,E6,BEFORE


In [ ]:
unique_docids = relations_df['docid'].unique()

docs_df = utils.get_docs(unique_docids)
docs_df.head()

In [ ]:
relations_df = utils.create_context_windows(relations_df, docs_df)
relations_df = utils.create_relation_labels(relations_df)
relations_df.head()

# Prepare Data for Training

In [4]:
tokenizer = models.create_temp_rel_tokenizer()

In [ ]:
# 90% train, 5% validation, 5% test (stratified by label)
train_df, temp_df = train_test_split(
    relations_df,
    test_size=0.1,
    random_state=42,
    stratify=relations_df["relation_id"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["relation_id"]
)

val_df = utils.augment_data(val_df)
test_df = utils.augment_data(test_df)

print(f"Train: {len(train_df)}")
print(f"Validation: {len(val_df)}")
print(f"Test: {len(test_df)}")

In [ ]:
train_loader = utils.create_data_loader(train_df, tokenizer, batch_size=16)
val_loader = utils.create_data_loader(val_df, tokenizer, batch_size=16)
test_loader = utils.create_data_loader(test_df, tokenizer, batch_size=16)

print("num training batches:", len(train_loader))

# Model Creation

In [ ]:
loss_fn = torch.nn.CrossEntropyLoss()

def train_one_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0.0

    all_preds = []
    all_labels = []

    for input_ids, attention_mask, labels in loader:
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        labels = labels.to(device)  # (B,) long

        # Keep only samples that contain both target markers
        has_t1 = (input_ids == model.t1_id).any(dim=1)
        has_t2 = (input_ids == model.t2_id).any(dim=1)
        valid = has_t1 & has_t2
        if not valid.any():
            continue

        input_ids = input_ids[valid]
        attention_mask = attention_mask[valid]
        labels = labels[valid]

        optimizer.zero_grad()
        logits = model(input_ids=input_ids, attention_mask=attention_mask)  # (B, C)
        loss = loss_fn(logits, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * input_ids.size(0)
        all_preds.append(torch.argmax(logits, dim=-1).cpu())
        all_labels.append(labels.cpu())

    y_pred = torch.cat(all_preds).numpy()
    y_true = torch.cat(all_labels).numpy()

    macro_f1 = f1_score(y_true, y_pred, average="macro")
    acc = accuracy_score(y_true, y_pred)

    return total_loss / len(loader.dataset), macro_f1, acc

@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    total_loss = 0.0

    all_preds = []
    all_labels = []

    for input_ids, attention_mask, labels in loader:
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        labels = labels.to(device)

        # Keep only samples that contain both target markers
        has_t1 = (input_ids == model.t1_id).any(dim=1)
        has_t2 = (input_ids == model.t2_id).any(dim=1)
        valid = has_t1 & has_t2
        if not valid.any():
            continue

        input_ids = input_ids[valid]
        attention_mask = attention_mask[valid]
        labels = labels[valid]

        logits = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = loss_fn(logits, labels)

        preds = torch.argmax(logits, dim=-1)

        total_loss += loss.item() * input_ids.size(0)
        all_preds.append(preds.cpu())
        all_labels.append(labels.cpu())

    y_pred = torch.cat(all_preds).numpy()
    y_true = torch.cat(all_labels).numpy()

    macro_f1 = f1_score(y_true, y_pred, average="macro")
    acc = accuracy_score(y_true, y_pred)

    return {
        "val_loss": total_loss / len(loader.dataset),
        "macro_f1": macro_f1,
        "accuracy": acc,
    }

In [ ]:
# Build model
num_labels = len(const.relation2id)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = models.TemporalRelationsModel(num_labels=num_labels, tokenizer=tokenizer).to(device)

# Cross-entropy setup
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

# First Stage: Basic Data Set

In [ ]:
# Train
epochs = 12
for epoch in range(epochs):
    train_loss, train_f1, train_acc = train_one_epoch(model, train_loader, optimizer, device)

    val_metrics = evaluate(model, val_loader, device)
    val_loss = val_metrics["val_loss"]
    val_f1 = val_metrics["macro_f1"]
    val_acc = val_metrics["accuracy"]

    print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} | Train F1: {train_f1:.4f} | Train Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} | Val F1: {val_f1:.4f} | Val Acc: {val_acc:.4f}")

# Second Stage: Augmented Data Set

In [ ]:
augmented_df = utils.augment_data(train_df, verbose=True)
augmented_loader = utils.create_data_loader(augmented_df, tokenizer, batch_size=16)

print("num training batches:", len(augmented_loader))

In [ ]:
# Train
# epochs = 6
# for epoch in range(epochs):
#     train_loss, train_f1, train_acc = train_one_epoch(model, augmented_loader, optimizer, device)

#     val_metrics = evaluate(model, val_loader, device)
#     val_loss = val_metrics["val_loss"]
#     val_f1 = val_metrics["macro_f1"]
#     val_acc = val_metrics["accuracy"]

#     print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} | Train F1: {train_f1:.4f} | Train Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} | Val F1: {val_f1:.4f} | Val Acc: {val_acc:.4f}")

# Analysis

In [ ]:
from sklearn.metrics import confusion_matrix

# Predict on test set
model.eval()
all_preds, all_true = [], []

with torch.no_grad():
    for input_ids, attention_mask, labels in test_loader:
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        labels = labels.to(device)

        # Keep only samples that contain both target markers
        has_t1 = (input_ids == model.t1_id).any(dim=1)
        has_t2 = (input_ids == model.t2_id).any(dim=1)
        valid = has_t1 & has_t2
        if not valid.any():
            continue

        input_ids = input_ids[valid]
        attention_mask = attention_mask[valid]
        labels = labels[valid]
        
        logits = model(input_ids=input_ids, attention_mask=attention_mask)
        preds = torch.argmax(logits, dim=-1).cpu()

        all_preds.append(preds)
        all_true.append(labels)

y_pred = torch.cat(all_preds).numpy()
y_true = torch.cat(all_true).detach().cpu().numpy()

# Confusion matrix (rows=true, cols=pred)
label_ids = [0, 1, 2, 3]
cm = confusion_matrix(y_true, y_pred, labels=label_ids)

cm_df = pd.DataFrame(
    cm,
    index=[f"true_{const.id2relation[i]}" for i in label_ids],
    columns=[f"pred_{const.id2relation[i]}" for i in label_ids],
)
display(cm_df)

test_metrics = evaluate(model, test_loader, device)
print(f"Test Loss: {test_metrics['val_loss']:.4f} | Test F1: {test_metrics['macro_f1']:.4f} | Test Acc: {test_metrics['accuracy']:.4f}")

# Saving model weights

In [ ]:
filename = 'temp_rel_roberta.pt'
torch.save(model.state_dict(), filename)
print(f"Model saved to '{filename}'")